In [1]:
import siibra
from nilearn import plotting, image
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import plotly.graph_objects as go

import pyNN.neuron as sim
from pyNN import space
from pyNN.random import RandomDistribution, NumpyRNG
from pyNN.neuron import Projection, StaticSynapse, FixedNumberPreConnector
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

import scipy.signal as signal

import pandas as pd

[siibra:INFO] Version: 1.0.1-alpha.10
[siibra:WARNING] This is a development release. Use at your own risk.
[siibra:INFO] Please file bugs and issues at https://github.com/FZJ-INM1-BDA/siibra-python.
/home/tkmcg/neuron-env/lib/python3.12/site-packages/pyNN/neuron/__init__.py:14: UserWarning: mpi4py not available
  warnings.warn("mpi4py not available")


In [2]:
from nrnutils import Mechanism, Section

class SimpleNeuron(object):

    def __init__(self, **parameters):
        hh = Mechanism('hh', gl=parameters['g_leak'], el=-65, gnabar=parameters['gnabar'], gkbar=parameters['gkbar'])
        self.soma = Section(L=30, diam=30, mechanisms=[hh])
        self.soma.add_synapse('ampa', 'Exp2Syn', e=0.0, tau1=0.1, tau2=5.0)
        self.initial_values = {"v": -65.0}

        self.soma.insert("extracellular")

        # needed for PyNN
        self.source_section = self.soma
        self.source = self.soma(0.5)._ref_v
        self.parameter_names = ('g_leak', 'gnabar', 'gkbar')
        self.traces = {}
        self.recording_time = False
        

In [3]:
from pyNN.neuron import NativeCellType

class SimpleNeuronType(NativeCellType):
    default_parameters = {'g_leak': 0.0002, 'gkbar': 0.036, 'gnabar': 0.12}
    default_initial_values = {'v': -65.0}
    recordable = ['soma(0.5).v', 'soma(0.5).ina']
    units = {'soma(0.5).v' : 'mV', 'soma(0.5).ina': 'nA'}
    receptor_types = ['soma.ampa']
    model = SimpleNeuron

In [4]:
##### Add in the STN neurons and change the geometry to match the coupling project with DC current ####
import pyNN.neuron as sim
import matplotlib.pyplot as plt
from pyNN import space
from pyNN.random import RandomDistribution, NumpyRNG
import numpy as np
from pyNN.neuron import Projection, StaticSynapse, FixedNumberPreConnector
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

v_init = -68
rng_seed = 856

############ Setup the simulation##########
sim.setup(timestep=0.01)

###########Set up the structure ############
cortical_space = space.RandomStructure(boundary=space.Sphere(2000))


########## create a population of neurons ###############
Cortical_Pop = sim.Population(100, SimpleNeuronType(), structure=cortical_space)

In [5]:
def collateral_positions(tgt_pop_z_coordinate, L, nseg):
    segment_centres = np.arange(0, nseg + 3 - 1) * (1 / nseg)
    segment_centres = segment_centres - (1 / (2 * nseg))
    segment_centres[0] = 0
    segment_centres[-1] = 1
    segment_centres = segment_centres[1 : len(segment_centres) - 1]
    z_coordinate = L * segment_centres - L / 2

    collateral_coordinates = np.array([])

    for cell in tgt_pop_z_coordinate:
        for z_distance in z_coordinate:            
            collateral_coordinates = np.append(collateral_coordinates, cell + z_distance)       
    
    return collateral_coordinates
    

In [6]:
x_points_ctx = Cortical_Pop.positions[0]
y_points_ctx = Cortical_Pop.positions[1]
z_points_ctx = Cortical_Pop.positions[2]


x_points_collaterals = np.repeat(x_points_ctx, 11)
y_points_collaterals = np.repeat(y_points_ctx, 11)
z_points_collaterals = collateral_positions(z_points_ctx, 10, 11)


In [20]:
voltage_file = pd.read_csv(
    "../MDT3389_Mesh2_050126_E3_BSGrid.txt",
    sep=r"\s+",
    header=None,
    names=["x", "y", "z", "V"]
)


x_voltage_coordinates = voltage_file['x'].values*1000
y_voltage_coordinates = voltage_file['y'].values*1000
z_voltage_coordinates = voltage_file['z'].values*1000-6250
voltages = voltage_file['V'].values*1000

In [21]:
points = np.array([x_voltage_coordinates,y_voltage_coordinates,z_voltage_coordinates]).T
Vint = LinearNDInterpolator(points, voltages)

In [39]:
interpolated_values_ctx = Vint(np.column_stack((x_points_collaterals, y_points_collaterals, z_points_collaterals)))

In [40]:
interpolated_values_ctx =  np.array(np.array_split(interpolated_values_ctx, 100))
print(interpolated_values_ctx)



[[751.04460628 750.99434715 750.94408802 ... 750.64253323 750.59227409
  750.54201496]
 [753.35584447 753.27051448 753.1851845  ... 752.67320459 752.58787461
  752.50254462]
 [799.27581373 799.12036029 798.96490685 ... 798.0321862  797.87673276
  797.72127932]
 ...
 [924.67089392 924.71879739 924.76670086 ... 925.05412167 925.10202514
  925.14992861]
 [819.34865831 819.38094204 819.41322577 ... 819.60692814 819.63921187
  819.6714956 ]
 [759.2408645  759.29856253 759.35626056 ... 759.70244873 759.76014676
  759.81784479]]
